<a href="https://colab.research.google.com/github/aytug-guler/GROUP19_CENG313/blob/main/assignment1_eda_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CENG313 - Assignment 1: EDA and Baseline Models

**Competition:** MABe Challenge - Social Action Recognition in Mice

**Group:** GROUP19 (Medium)

**Members:** Efe Zeybek, Aytuğ Güler

## Objective
This notebook explores the MABe mouse behavior dataset, analyzes its structure and distributions, and develops baseline models for social behavior recognition.


In [1]:

%pip install -q kagglehub pandas numpy matplotlib seaborn scikit-learn pyarrow


In [2]:

import os
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, precision_score, recall_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

print("Environment setup completed successfully!")
print("Pandas version:", pd.__version__)


Environment setup completed successfully!
Pandas version: 2.2.3


In [6]:

data_dir = Path(
    kagglehub.competition_download(
        "MABe-mouse-behavior-detection"
    )
)

print("Dataset directory:", data_dir)
print("\nFiles and folders:")

for item in data_dir.iterdir():
    print("-", item.name)


100%|██████████| 2.63G/2.63G [00:37<00:00, 74.6MB/s]

Extracting files...


Dataset directory: /root/.cache/kagglehub/competitions/MABe-mouse-behavior-detection

Files and folders:
- train_tracking
- train_annotation
- test.csv
- sample_submission.csv
- test_tracking
- train.csv


## 1. Exploratory Data Analysis (EDA)

In this section, we explore the structure, metadata, and characteristics of the MABe dataset before developing baseline models.

In [7]:

# Load training and test metadata

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

print("Training dataset shape:", train.shape)
print("Test dataset shape:", test.shape)

print("\nTraining columns:")
print(train.columns.tolist())

print("\nFirst five training records:")
display(train.head())

print("\nFirst five test records:")
display(test.head())


Training dataset shape: (8789, 38)
Test dataset shape: (1, 38)

Training columns:
['lab_id', 'video_id', 'mouse1_strain', 'mouse1_color', 'mouse1_sex', 'mouse1_id', 'mouse1_age', 'mouse1_condition', 'mouse2_strain', 'mouse2_color', 'mouse2_sex', 'mouse2_id', 'mouse2_age', 'mouse2_condition', 'mouse3_strain', 'mouse3_color', 'mouse3_sex', 'mouse3_id', 'mouse3_age', 'mouse3_condition', 'mouse4_strain', 'mouse4_color', 'mouse4_sex', 'mouse4_id', 'mouse4_age', 'mouse4_condition', 'frames_per_second', 'video_duration_sec', 'pix_per_cm_approx', 'video_width_pix', 'video_height_pix', 'arena_width_cm', 'arena_height_cm', 'arena_shape', 'arena_type', 'body_parts_tracked', 'behaviors_labeled', 'tracking_method']

First five training records:


,lab_id,video_id,mouse1_strain,mouse1_color,mouse1_sex,mouse1_id,mouse1_age,mouse1_condition,mouse2_strain,mouse2_color,...,pix_per_cm_approx,video_width_pix,video_height_pix,arena_width_cm,arena_height_cm,arena_shape,arena_type,body_parts_tracked,behaviors_labeled,tracking_method
0,AdaptableSnail,44566106,CD-1 (ICR),white,male,10.0,8-12 weeks,wireless device,CD-1 (ICR),white,...,16.0,1228,1068,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""head...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut
1,AdaptableSnail,143861384,CD-1 (ICR),white,male,3.0,8-12 weeks,NaN,CD-1 (ICR),white,...,9.7,968,608,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""late...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut
2,AdaptableSnail,209576908,CD-1 (ICR),white,male,7.0,8-12 weeks,NaN,CD-1 (ICR),white,...,16.0,1266,1100,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""late...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut
3,AdaptableSnail,278643799,CD-1 (ICR),white,male,11.0,8-12 weeks,wireless device,CD-1 (ICR),white,...,16.0,1224,1100,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""head...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut
4,AdaptableSnail,351967631,CD-1 (ICR),white,male,14.0,8-12 weeks,NaN,CD-1 (ICR),white,...,16.0,1204,1068,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""late...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut



First five test records:


,lab_id,video_id,mouse1_strain,mouse1_color,mouse1_sex,mouse1_id,mouse1_age,mouse1_condition,mouse2_strain,mouse2_color,...,pix_per_cm_approx,video_width_pix,video_height_pix,arena_width_cm,arena_height_cm,arena_shape,arena_type,body_parts_tracked,behaviors_labeled,tracking_method
0,AdaptableSnail,438887472,CD-1 (ICR),white,male,13.0,8-12 weeks,wireless device,CD-1 (ICR),white,...,16.0,1214,1090,60.0,60.0,square,familiar,"[""body_center"", ""ear_left"", ""ear_right"", ""head...","[""mouse1,mouse2,approach"", ""mouse1,mouse2,atta...",DeepLabCut


In [8]:

# Inspect tracking and annotation files

tracking_files = sorted(
    (data_dir / "train_tracking").rglob("*.parquet")
)

annotation_files = sorted(
    (data_dir / "train_annotation").rglob("*.parquet")
)

test_tracking_files = sorted(
    (data_dir / "test_tracking").rglob("*.parquet")
)

print("Training tracking files:", len(tracking_files))
print("Training annotation files:", len(annotation_files))
print("Test tracking files:", len(test_tracking_files))

if tracking_files:
    print("\nExample tracking file:")
    print(tracking_files[0])

if annotation_files:
    print("\nExample annotation file:")
    print(annotation_files[0])


Training tracking files: 8789
Training annotation files: 847
Test tracking files: 1

Example tracking file:
/root/.cache/kagglehub/competitions/MABe-mouse-behavior-detection/train_tracking/AdaptableSnail/1212811043.parquet

Example annotation file:
/root/.cache/kagglehub/competitions/MABe-mouse-behavior-detection/train_annotation/AdaptableSnail/1212811043.parquet
